# EDA - Exploratórna Analýza Dát
## Fraud Detection System

Tento notebook obsahuje detailný prieskum dát potrebný na pochopenie:
- Štruktúry a kvality dát
- Rozdelenia fraud prípadov
- Vzťahov medzi premennými
- Anomálií a chýb v dátach

---

In [30]:
import pandas as pd
import numpy as np
from typing import List, Optional
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

# Nastavenie štýlu pre lepšie grafiky
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# EDA - Exploratórna Analýza Dát
## Fraud Detection System

Tento notebook obsahuje detailný prieskum dát potrebný na pochopenie:
- Štruktúry a kvality dát
- Rozdelenia fraud prípadov
- Vzťahov medzi premennými
- Anomálií a chýb v dátach

In [ ]:
# =============================================================================
# 1. NAČÍTANIE DÁTOVÝCH ZDROJOV
# =============================================================================
# Načítavame tri hlavné tabuľky:
# - cards: informácie o kartách
# - transactions: jednotlivé transakcie
# - users: informácie o používateľoch

root = Path.cwd().parent
data_dir = root / "data" / "raw"

print("Nacitavam data...")
cards = pq.read_table(data_dir / "cards.pq").to_pandas()
transactions = pq.read_table(data_dir / "transactions.pq").to_pandas()
users = pq.read_table(data_dir / "users.pq").to_pandas()
print("Data nacitana uspesne!")

Nacitavam data...
Data nacitana uspesne!


## Fáza 1: Základné Informácie o Dátach

In [ ]:
# =============================================================================
# 2. ROZMERY A ŠTRUKTÚRA DATASETU
# =============================================================================

print("ROZMERY DATASETU")
print("=" * 60)
print(f"Transakcie:    {transactions.shape[0]:>10} riadkov, {transactions.shape[1]:>3} stĺpcov")
print(f"Používatelia:  {users.shape[0]:>10} riadkov, {users.shape[1]:>3} stĺpcov")
print(f"Karty:         {cards.shape[0]:>10} riadkov, {cards.shape[1]:>3} stĺpcov")
print("=" * 60)

# Vytvoríme dátum zo stĺpcov Year / Month / Day, pretože priamy stĺpec Date v datasetu neexistuje.
transactions['Date'] = pd.to_datetime(
    transactions[['Year', 'Month', 'Day']].rename(columns={'Year': 'year', 'Month': 'month', 'Day': 'day'}),
    errors='coerce'
)
transactions['Amount'] = transactions['Amount'].str.replace('$','').astype(float)

print(f"\nCasovy rozsah transakcií: {transactions['Date'].min()} - {transactions['Date'].max()}")
print(f"Suma transakcií: ${transactions['Amount'].min():.2f} - ${transactions['Amount'].max():.2f}")

ROZMERY DATASETU
Transakcie:       3782053 riadkov,  15 stĺpcov
Používatelia:        2000 riadkov,  19 stĺpcov
Karty:               6146 riadkov,  13 stĺpcov


KeyError: 'Date'

In [ ]:
# =============================================================================
# 3. DÁTOVÉ TYPY A PRVÝ POHĽAD NA DÁTA
# =============================================================================

print("DÁTOVÉ TYPY - TRANSAKCIE\n")
print(transactions.dtypes)

print("\n\n--- Prvých 5 transakcií ---")
display(transactions.head())

print("\n\nDÁTOVÉ TYPY - POUŽÍVATELIA\n")
print(users.dtypes)

print("\n\n--- Prvých 5 používateľov ---")
display(users.head())

print("\n\nDÁTOVÉ TYPY - KARTY\n")
print(cards.dtypes)

print("\n\n--- Prvých 5 kariet ---")
display(cards.head())

## Fáza 2: Kontrola Kvality Dát

## Fáza 2: Kontrola Kvality Dát

In [ ]:
# =============================================================================
# 4. CHÝBAJÚCE HODNOTY
# =============================================================================

print("CHÝBAJÚCE HODNOTY (NULL)")
print("=" * 80)

for df_name, df in [('TRANSAKCIE', transactions), ('POUŽÍVATELIA', users), ('KARTY', cards)]:
    print(f"\n{df_name}:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("  Žiadne chýbajúce hodnoty")
    else:
        for col in missing[missing > 0].index:
            pct = (missing[col] / len(df)) * 100
            print(f"  {col:30s}: {missing[col]:>7} ({pct:>5.2f}%)")

In [ ]:
# =============================================================================
# 5. DUPLICITNÉ ZÁZNAMY
# =============================================================================

print(" DUPLICITNÉ ZÁZNAMY")
print("=" * 80)

# Kontrola úplne duplicitných riadkov
dup_trans = transactions.duplicated().sum()
dup_users = users.duplicated().sum()
dup_cards = cards.duplicated().sum()

print(f"Transakcie - úplne duplikáty:  {dup_trans} ({(dup_trans/len(transactions))*100:.2f}%)")
print(f"Používatelia - úplne duplikáty: {dup_users} ({(dup_users/len(users))*100:.2f}%)")
print(f"Karty - úplne duplikáty:        {dup_cards} ({(dup_cards/len(cards))*100:.2f}%)")

# Kontrola kľúčových polí
print("\n--- Unikátnosť kľúčových polí ---")
print(f"Unikátni používatelia:    {users['User'].nunique():>5} z {len(users):>5}")
print(f"Unikátne karty (index):   {cards['CARD INDEX'].nunique():>5} z {len(cards):>5}")
print(f"Unikátne karty v transc.: {transactions['Card'].nunique():>5} z {len(transactions):>5}")

#  UPOZORNENIE: Je len 9 unikátnych CARD INDEX!
print("\n  UPOZORNENIE: Len 9 unikátnych hodnôt v 'CARD INDEX'!")
print("    To znamená, že tieto 9 hodnôt sa opakujú pre rôznych používateľov.")

## Fáza 3: Analýza TARGET Premennej (Fraud)

In [ ]:
# =============================================================================
# 6. DISTRIBÚCIA FRAÚD PRÍPADOV
# =============================================================================

print("ANALÝZA FRAÚD PRÍPADOV")
print("=" * 80)

fraud_counts = transactions['Is Fraud?'].value_counts(dropna=False)
fraud_pcts = (transactions['Is Fraud?'].value_counts(normalize=True, dropna=False) * 100)

fraud_df = pd.DataFrame({
    'Počet': fraud_counts,
    'Percento': fraud_pcts.round(2)
})

print("\nRozdelenie fraud/non-fraud transakcií:")
print(fraud_df)

print(f"\nPOMER:")
fraud_ratio = fraud_counts.get('Yes', 0) / fraud_counts.get('No', 1)
print(f"   {fraud_counts.get('Yes', 0)} fraud : {fraud_counts.get('No', 0)} normálnych")
print(f"   Ratio: 1 : {1/fraud_ratio:.1f}" if fraud_ratio > 0 else "   Žiadny fraud")

# Vizualizácia
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
fraud_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Distribúcia Fraúd Prípadov', fontsize=14, fontweight='bold')
ax.set_ylabel('Počet transakcií')
ax.set_xlabel('Je Fraúd?')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print("\nVAROVANIE: Dáta sú VYSOKO NEVYVÁŽENÉ (imbalanced)!")
print("    -> Pri modelovaní budeme musieť použiť stratégie na balancovanie")

## Fáza 4: Analýza Kategorických Premenných

In [ ]:
# =============================================================================
# 7. ANALÝZA KATEGORICKÝCH PREMENNÝCH V TRANSAKCIÁCH
# =============================================================================

print("KATEGORICKÉ PREMENNÉ V TRANSAKCIÁCH")
print("=" * 80)

categorical_cols = ['Type', 'Errors?', 'MCC', 'Merchant State', 'Merchant City']

for col in categorical_cols:
    if col in transactions.columns:
        print(f"\n{col}: {transactions[col].nunique()} unikátnych hodnôt")
        print("Top 5:")
        print(transactions[col].value_counts().head(5))
        print()

# Analýza vzťahu medzi Type a Fraud
print("\n" + "=" * 80)
print("VZŤAH: Transaction Type vs Fraud")
print("=" * 80)
type_fraud = pd.crosstab(transactions['Type'], transactions['Is Fraud?'], margins=True)
print(type_fraud)

fraud_rate_by_type = pd.crosstab(transactions['Type'], transactions['Is Fraud?'], normalize='index') * 100
print("\nFraud rate (%) podľa typu transakcie:")
print(fraud_rate_by_type.round(2))

In [ ]:
# =============================================================================
# 8. ANALÝZA NUMERICKÝCH PREMENNÝCH
# =============================================================================

print("NUMERICKÉ PREMENNÉ - ŠTATISTIKA")
print("=" * 80)

numeric_cols = transactions.select_dtypes(include=[np.number]).columns

print("\nZÁKLADNÉ ŠTATISTIKY:")
print(transactions[numeric_cols].describe().to_string())

# Analýza Amount vzťahu s Fraud
print("\n\nANALÝZA: Amount vs Fraud")
print("=" * 80)
for fraud_status in ['No', 'Yes']:
    amounts = transactions[transactions['Is Fraud?'] == fraud_status]['Amount']
    print(f"\n{fraud_status} - {len(amounts)} transakcií:")
    print(f"  Priemer: ${amounts.mean():.2f}")
    print(f"  Medián:  ${amounts.median():.2f}")
    print(f"  Min:     ${amounts.min():.2f}")
    print(f"  Max:     ${amounts.max():.2f}")
    print(f"  Std:     ${amounts.std():.2f}")

# Vizualizácia Amount distribúcie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bez fraúdu
axes[0].hist(transactions[transactions['Is Fraud?'] == 'No']['Amount'], bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[0].set_title('Amount - Normálne Transakcie', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Suma ($)')
axes[0].set_ylabel('Počet')

# S fraúdom
axes[1].hist(transactions[transactions['Is Fraud?'] == 'Yes']['Amount'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].set_title('Amount - Fraúdne Transakcie', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Suma ($)')
axes[1].set_ylabel('Počet')

plt.tight_layout()
plt.show()

## Fáza 5: Spájanie a Integracia Dát

In [ ]:
# =============================================================================
# 9. SPÁJANIE TABULIEK (MERGE)
# =============================================================================

print("\nSPÁJANIE TABULIEK...")
print("Spojujeme: Transakcie + Používatelia + Karty")

# SPRÁVNY SPÔSOB: Spájať na USER a CARD INDEX
# Dôvod: CARD INDEX má len 9 unikátnych hodnôt a musí byť spojený s konkrétnym User
transactions_full = (
    transactions
    .merge(users, on='User', how='left')
    .merge(cards,
           left_on=['User', 'Card'],
           right_on=['User', 'CARD INDEX'],
           how='left')
)

print(f"Spájanie hotovo!")
print(f"   Výsledný dataset: {transactions_full.shape[0]} riadkov, {transactions_full.shape[1]} stĺpcov")
print(f"   Chýbajúce hodnoty: {transactions_full.isnull().sum().sum()} NULL hodnôt celkom")

print("\nStĺpce v zjednotenom datasete:")
print(transactions_full.columns.tolist())

print("\nPrvých 5 riadkov zjednotených dát:")
display(transactions_full.head())

## Fáza 6: Zhrnutie a Odporúčania Pre Preprocessing

In [ ]:
# =============================================================================
# 10. ZHRNUTIE ZISTENÍ Z EDA
# =============================================================================

print("\n" + "=" * 80)
print("ZHRNUTIE EXPLORATÓRNEJ ANALÝZY")
print("=" * 80)

print("""
KVALITA DÁT:
   - Žiadne chýbajúce hodnoty (NULL)
   - Žiadne úplne duplicitné riadky
   - Všetky dáta sú korektne načítané

PROBLÉM - NEVYVÁŽENÉ DÁTA (CLASS IMBALANCE):
   - Fraud: {:.2f}% z celkových transakcií
   - Non-fraud: {:.2f}% z celkových transakcií
   -> Budeme musieť použiť SMOTE, undersampling, alebo váhované metriky

KLÚČOVÉ ZISTENIA:
   1. Amount má rozdielnu distribúciu pre fraud vs non-fraud
   2. Niektoré typy transakcií majú vyššiu fraud rate
   3. Geografické premenné (Merchant State, City) majú potenciál
   4. FICO Score a Yearly Income sú dôležité pre profiling

NASLEDUJÚCE KROKY V PREPROCESSING:
   1. Normalizácia numerických premenných (Amount, Age, Income, FICO)
   2. Kódovanie kategorických premenných (One-hot encoding)
   3. Tvorba feature engineering premenných
   4. Balancovanie trénovacieho datasetu
   5. Rozdelenie na train/test sady

DÁTA PRE MODEL:
   - Počet riadkov: {rows}
   - Počet stĺpcov: {cols}
   - Cieľová premenná: Is Fraud? (Yes/No)
""".format(
    (fraud_counts.get('Yes', 0) / len(transactions) * 100),
    (fraud_counts.get('No', 0) / len(transactions) * 100),
    rows=transactions_full.shape[0],
    cols=transactions_full.shape[1]
))

print("=" * 80)
print("EDA HOTOVÁ - POKRAČUJEME V PREPROCESSING NOTEBOOKU")